# Tuning Analysis from a Real Optuna Study

This notebook analyzes the output of a **real Optuna-based VAMOS tuning run** instead of synthetic data.

We will:
1. generate a compact Optuna study,
2. convert the trial history into a DataFrame,
3. inspect convergence,
4. analyze how each hyperparameter relates to the score.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vamos import optimize
from vamos.algorithms import NSGAIIConfig
from vamos.engine.tuning import (
    EvalContext,
    Instance,
    Int,
    ModelBasedTuner,
    ParamSpace,
    Real,
    TuningTask,
    available_model_based_backends,
    filter_active_config,
)
from vamos.foundation.quality_indicators import compute_hypervolume

if not available_model_based_backends().get('optuna'):
    raise RuntimeError("Optuna is not available. Install vamos-optimization[tuning].")


## 1. Run a Small Optuna Study

We keep the search space compact here so the analysis is easy to interpret.


In [ ]:
REF_POINT = np.array([1.1, 1.1])

param_space = ParamSpace(
    params={
        'pop_size': Int('pop_size', 60, 140),
        'crossover_prob': Real('crossover_prob', 0.70, 0.98),
        'crossover_eta': Real('crossover_eta', 8.0, 35.0),
        'mutation_eta': Real('mutation_eta', 8.0, 35.0),
    }
)


def eval_fn(config: dict, ctx: EvalContext) -> float:
    cfg = (
        NSGAIIConfig.builder()
        .pop_size(int(config['pop_size']))
        .selection('tournament', size=2)
        .crossover('sbx', prob=float(config['crossover_prob']), eta=float(config['crossover_eta']))
        .mutation('polynomial', prob='1/n', eta=float(config['mutation_eta']))
        .build()
    )
    result = optimize(
        'zdt1',
        algorithm='nsgaii',
        algorithm_config=cfg,
        max_evaluations=1200,
        seed=ctx.seed,
        n_var=20,
        engine='numpy',
    )
    return float(compute_hypervolume(result.F, REF_POINT))


task = TuningTask(
    name='optuna_history_analysis',
    param_space=param_space,
    instances=[Instance(name='zdt1', n_var=20, kwargs={})],
    seeds=[0, 1],
    budget_per_run=1200,
    maximize=True,
    aggregator=np.mean,
)

tuner = ModelBasedTuner(
    task=task,
    max_trials=12,
    backend='optuna',
    optuna_sampler='tpe',
    seed=23,
    n_jobs=1,
)

best_config, history = tuner.run(eval_fn, verbose=False)
print('Best configuration:', best_config)


## 2. Build a Trial DataFrame

Each row represents one Optuna trial, with the score and the active hyperparameters used in that trial.


In [ ]:
rows = []
for trial in history:
    rows.append(
        {
            'trial_id': trial.trial_id,
            'score': trial.score,
            **filter_active_config(trial.config, task.param_space),
        }
    )

df = pd.DataFrame(rows).sort_values('trial_id').reset_index(drop=True)
df


## 3. Convergence Trace

This plot answers the first question: did the study keep finding better configurations over time?


In [ ]:
df['best_so_far'] = df['score'].cummax()

plt.figure(figsize=(9, 4.5))
plt.plot(df['trial_id'], df['score'], 'o-', alpha=0.65, label='trial score')
plt.plot(df['trial_id'], df['best_so_far'], linewidth=2.5, label='best so far')
plt.xlabel('Trial')
plt.ylabel('Hypervolume')
plt.title('Optuna tuning trace on ZDT1')
plt.legend()
plt.tight_layout()
plt.show()


## 4. Parameter vs Score Views

Scatter plots are the fastest way to check whether a parameter looks directional, flat, or noisy.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(df['pop_size'], df['score'], s=70, alpha=0.75)
axes[0].set_xlabel('pop_size')
axes[0].set_ylabel('Hypervolume')
axes[0].set_title('Population size')

axes[1].scatter(df['crossover_prob'], df['score'], s=70, alpha=0.75)
axes[1].set_xlabel('crossover_prob')
axes[1].set_ylabel('Hypervolume')
axes[1].set_title('Crossover probability')

axes[2].scatter(df['mutation_eta'], df['score'], s=70, alpha=0.75)
axes[2].set_xlabel('mutation_eta')
axes[2].set_ylabel('Hypervolume')
axes[2].set_title('Mutation eta')

plt.tight_layout()
plt.show()


## 5. Rank Trials and Compare the Top Slice

Looking only at the top few trials often makes the promising parameter region easier to see.


In [ ]:
ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
top_slice = ranked.head(5)

display_cols = ['trial_id', 'score', 'pop_size', 'crossover_prob', 'crossover_eta', 'mutation_eta']
top_slice[display_cols]


## 6. Numeric Correlations

This is a quick proxy for parameter importance. It is not causal, but it helps identify which knobs are worth deeper follow-up runs.


In [ ]:
numeric_cols = ['score', 'pop_size', 'crossover_prob', 'crossover_eta', 'mutation_eta']
corr = df[numeric_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.to_numpy(), cmap='coolwarm', vmin=-1.0, vmax=1.0)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title('Correlation matrix')
fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

corr['score'].sort_values(ascending=False)


## Next Steps

- Continue with [33_optuna_tuning_advanced.ipynb](./33_optuna_tuning_advanced.ipynb) for multi-fidelity Optuna studies and persistent storage.
- Use the exported-history workflow from [21_programmatic_tuning.ipynb](./21_programmatic_tuning.ipynb) when you want to analyze studies outside the notebook.
